# Lab 5 &ndash; Discrete logarithms

As usual, start by writing <span style="color: red">HACKOOLIQUES</span> here.



## A) Brute force

Here is a small (20 bits: very unsafe!) prime modulus that we will use for demonstration purposes.

In [6]:
n = 668579

In [6]:
from is_prime import *  # from RSA lab

is_prime(n)

False

By brute-forcing the exponent, compute $\mathrm{dlog}_n(128032,539188)$, that is: 

$$ \text{find a } \ell \text{ such that } 539188^{\ \ell} \underset{n}{\equiv} 128032. $$

In [15]:
def brute_force_dlog(g,n,a,max_l=None):
    max_l = max_l if max_l else n
    for l in range(n):
        if l > max_l: return
        if pow(g,l,n) == a:
            return a

In [10]:
print(f"l: {brute_force_dlog(539188, n, 128032)}")

l: 128032


## B) Baby-step giant-step

Let us now move to a slightly larger (still very much unsafe!) 40-bit modulus:

In [38]:
n = 508734848761

Suppose we want to solve the following instance of the DLP:

$$ \text{find } \ell \text{ such that } 232583746145^{\ \ell} \underset{n}{\equiv} 183352143085. $$

Start doing a brute force search for $\ell$ (up to, say, $2^{20}$) to see how fast it goes. How long do you think it would take you to find $\ell$ this way?

In [7]:
print(f"l: {brute_force_dlog(232583746145, n, 183352143085, pow(2,20))}")


NameError: name 'brute_force_dlog' is not defined

6.7 secondes pour 20 bits -> 40 bits = 20bits * 2^20 -> 6.7s * 2^20 (1048576) = 81j  ?

It should, however, be quite reasonable to find the exponent by precomputing (and storing) $2^{20}$ baby steps and performing as many giant steps as needed. What value of $\ell$ do you find (and how long did it take you)?

In [52]:
def step_dlog(g, n, a):
    from math import isqrt

    m = isqrt(n - 1) + 1

    # Baby steps: g^j
    baby = {}
    value = 1
    for j in range(m):
        if value not in baby:   # garde le plus petit j
            baby[value] = j
        value = (value * g) % n

    # Giant step factor: g^(-m)
    factor = pow(g, -m, n)

    # Giant steps
    value = a
    for i in range(m):
        if value in baby:
            exp = i * m + baby[value]
            # cas particulier: a == 1 → chercher l'ordre, pas juste 0
            if exp != 0:
                return exp
        value = (value * factor) % n

    return None

    
step_dlog(232583746145, n, 183352143085)
    

66416395796

## C) Chinese remainder theorem

For the 40-bit modulus above, the situation is actually much better for the attacker (you) since $n$ was foolishly chosen to be a composite number...

In [9]:
is_prime(n)

False

You should have no problem brute-forcing a factorization of $n$ to get much smaller integers $n_1$ and $n_2$ such that $n = n_1 \cdot n_2$.

In [10]:
def factoriser(n: int):
    facteurs = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            facteurs.append(d)
            n //= d
        d += 1
    if n > 1:
        facteurs.append(n)
    return facteurs

facteurs = factoriser(n)
import math

print(math.prod(facteurs) == n, facteurs)

True [551927, 921743]


Using this knowledge, you should be to recover the value of $\ell$ by:</p>

- breaking the congruence $g^\ell \underset{n}{\equiv} x$ into a system of two congruences $g^\ell \underset{n_i}{\equiv} x$;

- solving the DLP modulo each $n_i$, yielding congruences $\ell \underset{\mathrm{ord}_{n_i}(g)}{\equiv} \ell_i$;

- using the Chinese Remainder Theorem to recover the value of $\ell$ modulo the multiplicative order $\mathrm{ord}_n(g) = \mathrm{LCM}(\mathrm{ord}_{n_1}(g),\mathrm{ord}_{n_2}(g))$ of $g$ mod $n$.


In [36]:
import math
from functools import reduce

def lcm(a, b):
    return abs(a * b) // math.gcd(a, b)

def lcm_list(lst):
    return reduce(lcm, lst)

def crt(a1, m1, a2, m2):
    g = math.gcd(m1, m2)
    if (a1 - a2) % g != 0:
        return None
    # Simplified version for coprime moduli
    M = m1 * m2
    inv_m1 = pow(m1, -1, m2)
    x = (a2 - a1) * inv_m1 % m2
    x = (a1 + m1 * x) % M
    return x


li = l [mod ni]
li + m oi = l
li + m oi = lj [mod oj]
m = (lj - li)/ oi [mod oj]
==> li + m oi = l ==> l = li + (lj - li)/ oi


In [ ]:
import time
g = 232583746145
a = 183352143085
n = 508734848761
t0 = time.time()

l_classique = step_dlog(g, n, a)
print("l:",l_classique)
print(pow(g,l_classique,n))
print("Classique:", time.time() - t0)

t0 = time.time()
n1,n2=facteurs
l1, l2 = [step_dlog(g, n, a)for n in [n1,n2]]

o1, o2 = [step_dlog(g,n,1) for n in [n1,n2]]

inv_o1 = pow(o1, -1, o2)
m = ((l2-l1)*inv_o1)%o2
l=crt(l1, o1, l2, o2)
print("l:", l)
print("Restes chinois:", time.time() - t0)

l: 66416395796
183352143085
Classique: 0.7504353523254395
104623 True
275986 True
275963 1
460871 1
l: 66416395796
Restes chinois: 0.0020089149475097656


28874483278

## D) A longer modulus

Here is a much longer (4096 bits) prime modulus:

In [32]:
n = 0xea118cad53b34ee155158a978a0c0529fdb64e376fec81de062366deb30f7db7ea33f0f2e0283cf52cc19e0e9603282b6c2c090496e2c8cbd9f8a9d35a587c6a13813544611af5d67f576fe100fafa2489ad9d7858bca8fa97b428ad167d7817ea028a40b07a419063bd17a4139e6f6e8ded3e03b4c6ce10eee6257734a860f97107f702f804eb381887d5f29dccf60f26e5481b838fc39f460b63d308375ab66f7f74988ce4c493dd2c5f1a45072d29458d50dd9a317ddaa4e048e2a7d4b2bda8d8baa4c9ae59a4b4c2c85ddf21945b8e64f67b99fdc4cb7a7a361d572d840b92ccb2b2877e46c288267d9d2f8bc34d264f4f89911a5c3a59d7cc600ff7e5d4856ceac88c327f92f70ff848909064ad567e4e68fdbddc10edf0ae9d4849de2aaff2799966f00cc36dc18d77126edb66f745f3995fe049b4d1d4745f9c3e0a008cd75a59840a2ab31c9621e860f913ca667f7af605a9af4f710483a0663aa61510495554c4e6fc8211e438d94320b99da5ee6bd669be967baa4617feca2e8f7a84eb67fd7b9c3c6a9863d44595c94c7ca83303c53b395e4c976a92a44c190dc433cf7d1daf7927d2eef3f023058dc8a124ddabafd3c1141ee5b57cb6abdb31d533cfea4be06055ebd385de7e65f9a37b4a473c57e9592d713ee14ff2cdb9dfc18e0c45ded3082c367a46eb4573918cbd528e67436f2271fe4ce29467b76e57e9

In [27]:
is_prime(n)

True

and here is a good-looking value of $g$:

In [28]:
g = 0xcc76efe7a690cdf1d359d1098e6969380b1a164bbbc04a8a7cec5c0677462b2830f378dcb7fba42b43611afecca9feda39f315bc1f858f89ca17d7dc4657afab1857ae007b3dd875055fd07a8fe5f20dd84dc2dc0a21bf7f074bb54fd4f17797e35ee886cb776fd8ab054a238b4951fb8dccc723e2072e5309ca24fda7933fe25fff501a65a07753fef3d65824be4aef77b5f6f0b8bb222f442fb8e73f729b369308a05c5d951af70cb222abb41a465d29d42482dfbda06383e9d8a6486eff014e29543f705ee5adf515149ca55b6998dcc67dd6e27545d3fe26c605c9e698ac8769a310499ee4801201c7e11e11276a62efd9a01ed2dc77eda84b9563b92df4fc110f5c306bd7aa96c3bfa1ca2c3cc93d1be6d436fd7b4ce2f6cc2a014a10e844a1f498fc2d65b060e61d803769ca414a1a20d78f824dd256bd7f534e02d47c75ac4a4a429fc02e0c67b613162209d0769a70a05332331e735d703a9781154452bcacb21c9dd51c10b86a368df49f38da9876e0321a9812d9912536585a0ae9d45f8dad414d9faf60fe1824c22a420e81175c452cb6a5795ecaed2e01f0dc0c2aa9ee518caa5ca0221bee61b72d4e236cb213bf1e5fcd3123df896f3cbb47bedf04fbf458dc4d13dce75ef785fde6eda3f13e9285784a9103b1e7e1b14bfc8078bb157838ba34a8dc396d85d031808edcc150b37f353828cae6493d952a896e

Despite the appearances, computing mod $n$ discrete logarithms in base $g$ is pretty easy. Can you explain why?

Même avec un grand module, si n n’est pas choisi correctement (composite ou faible facteur premier), le logarithme discret peut être vulnérable.

In [35]:
q = (n-1) // 2
is_prime(q)

False

(n-1) // 2 n'est pas un nombre premier de Sophie Germain donc n n'est pas un premier sécurisé.

Convince me by deducing the shared secret between Alice and Bob after the following Diffie-Hellman exchange using public parameters $n$ and $g$.

Alice: 

`0x7ed3f900379002077bf053e5d6414692ba9ebca83aa1ab8568d582eaadc72aef8f84310f88e71a2295e5841755da7cadc2a249f9c3dc61e2979ad7e91f86b57114e8ac0616d8f869da2f65f98a7092c7a3d9de95b0be6c563dcd4bdf62ed1e4e1ebd3ea558296ad900130862db654b104ea19279cb152775836cd203991f396585b3a2da23467d23ac2a9ab788b87468558bd4b6ab2d063f7f3ba318dc4b4b43105bfbd6d2103b6ff6f1962b99ab52e18c9b8c6d751ad587445589e3dfecd9ea763623590168e4d40cfd6ebc227410795ed3bba882a6e6f147bf1c9fcbfec40e45653e78b2e6b4edf6c685d2994f46ed8c5b71fd1aac5a81cefea0ddb2301f0783aa38827ae16e5933bed6da8e8aaecfd75106a762836da01926962c3925c6170bce8d979cdbf64fba704a1a160cd4940a1e61b782b1e17e8281dfc99a604707f6799b6fbbcb4c1fae5e1a77fab1f2b3e16ade0e2ae436672c4f51456443cdb33dd84415828c8fcbf00e46b1aa0c3e4b8aec9dbe6b6a5bf9b08355cb73cb778e1208a1eab97bd83775e870ed933f9b999014ca99d64e8e3c7ed6485928b64047c323290c60a9854bac9046e3da8e801cf43efb4551d24383c3534e0ac03b55bb3968b228c21b64d898213b264893db08cb3b6e4ea252f01d30821654fc464e655327c65c2f1c4b8a68f7314fa70d635ac9bc03cb12b548e63b45cc2fe81471c6`

Bob: 

`0x1f97ecd409eb5bafad132e7989e1f972a0dbcb7013d327d72bf1869719a383ca80cadd23230ac6de827971ae93f716287ad115c43f9dd7acf6e7e4964357c2a992c53431de2f5fda4175ca19df93c16019d3cce8eb62f68a1fb368c214f7518674ac073b4f972ea1b446116844d5290ec8cc07e32745aef5012df625c93edce108e632d1776002cae80355b7aacf1842863f3a4fd4e86d9a035290aa65d3b454b3ae59e5301a71a81c2c2b814df511126c3db5d5034406bbeb593c0ddf71f1d90cb965ce6b7d74e096cd24297e6ef5c66105c43c14911e569b19faa3bd8d0222af2d831f0fa56eb266169065f3e7bf4b7371236f097869403ad73cb5cdb239c4004bb43c6851bda0dde7446da7efb46ca231cff7adbd299211f16a0ed528989a0146ccd49f62b3fa64eb00afeecfcb8be7a0a78f648676f03b2ab5bc5a8656e6b699e4bb0cd64d23a33f880ee1b8b957bfd8bee1f861e3a82bb12746ab0f11498877602d867e46e45b26e4c583b4ef28db91e4cc198342d13f8e05beb33f3896732f102ee5380a6fb2d30339082b2bfce82e1921d2010f21ceda34b3b82625069086f2fd65b53f547acbd0e028d85c3119184699333383c2907872a837722cfb8c38de59f83f00397a021cd052b9bf7b691b2222ae1ef4fb71040dbda9ed1c0bf58b4ad4872080a7dbdd245390d05cfca073efbc022916137da44f6005e09725`

In [30]:
a = 0x7ed3f900379002077bf053e5d6414692ba9ebca83aa1ab8568d582eaadc72aef8f84310f88e71a2295e5841755da7cadc2a249f9c3dc61e2979ad7e91f86b57114e8ac0616d8f869da2f65f98a7092c7a3d9de95b0be6c563dcd4bdf62ed1e4e1ebd3ea558296ad900130862db654b104ea19279cb152775836cd203991f396585b3a2da23467d23ac2a9ab788b87468558bd4b6ab2d063f7f3ba318dc4b4b43105bfbd6d2103b6ff6f1962b99ab52e18c9b8c6d751ad587445589e3dfecd9ea763623590168e4d40cfd6ebc227410795ed3bba882a6e6f147bf1c9fcbfec40e45653e78b2e6b4edf6c685d2994f46ed8c5b71fd1aac5a81cefea0ddb2301f0783aa38827ae16e5933bed6da8e8aaecfd75106a762836da01926962c3925c6170bce8d979cdbf64fba704a1a160cd4940a1e61b782b1e17e8281dfc99a604707f6799b6fbbcb4c1fae5e1a77fab1f2b3e16ade0e2ae436672c4f51456443cdb33dd84415828c8fcbf00e46b1aa0c3e4b8aec9dbe6b6a5bf9b08355cb73cb778e1208a1eab97bd83775e870ed933f9b999014ca99d64e8e3c7ed6485928b64047c323290c60a9854bac9046e3da8e801cf43efb4551d24383c3534e0ac03b55bb3968b228c21b64d898213b264893db08cb3b6e4ea252f01d30821654fc464e655327c65c2f1c4b8a68f7314fa70d635ac9bc03cb12b548e63b45cc2fe81471c6
b=0x1f97ecd409eb5bafad132e7989e1f972a0dbcb7013d327d72bf1869719a383ca80cadd23230ac6de827971ae93f716287ad115c43f9dd7acf6e7e4964357c2a992c53431de2f5fda4175ca19df93c16019d3cce8eb62f68a1fb368c214f7518674ac073b4f972ea1b446116844d5290ec8cc07e32745aef5012df625c93edce108e632d1776002cae80355b7aacf1842863f3a4fd4e86d9a035290aa65d3b454b3ae59e5301a71a81c2c2b814df511126c3db5d5034406bbeb593c0ddf71f1d90cb965ce6b7d74e096cd24297e6ef5c66105c43c14911e569b19faa3bd8d0222af2d831f0fa56eb266169065f3e7bf4b7371236f097869403ad73cb5cdb239c4004bb43c6851bda0dde7446da7efb46ca231cff7adbd299211f16a0ed528989a0146ccd49f62b3fa64eb00afeecfcb8be7a0a78f648676f03b2ab5bc5a8656e6b699e4bb0cd64d23a33f880ee1b8b957bfd8bee1f861e3a82bb12746ab0f11498877602d867e46e45b26e4c583b4ef28db91e4cc198342d13f8e05beb33f3896732f102ee5380a6fb2d30339082b2bfce82e1921d2010f21ceda34b3b82625069086f2fd65b53f547acbd0e028d85c3119184699333383c2907872a837722cfb8c38de59f83f00397a021cd052b9bf7b691b2222ae1ef4fb71040dbda9ed1c0bf58b4ad4872080a7dbdd245390d05cfca073efbc022916137da44f6005e09725

## E) Diffie-Hellman

Let's now start doing things properly and help Alice and Bob set up a secure shared secret. 

- Pick up some standard values of $n$ and $g$ for a 4096-bit modulus (e.g. the one <a href="https://www.ietf.org/rfc/rfc3526.txt">specified here</a>)
- verify that $n = 2 q + 1$ is a safe prime and $g$ an element of multiplicative order $q$ or $2q$ mod $n$
- and then play out the process of using the Diffie-Hellman key exchange protocol to set up a shared secret between Alice and Bob.

In [ ]:
# Alice
A = pow(g, a, n)

# Bob
B = pow(g, b, n)

# Shared secret
secret_alice = pow(B, a, n)
secret_bob = pow(A, b, n)

print(secret_alice == secret_bob)


True


Même avec de grands nombres, le DH reste sûr si n et g sont correctement choisis. Le logarithme discret devient difficile et le secret partagé est protégé.